# Browser automation with LlamaIndex

## Overview

This notebook demonstrates Amazon Bedrock AgentCore browser automation with LlamaIndex.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM model           | Amazon Bedrock Claude 3 Haiku                                                   |
| Tutorial components | Using LlamaIndex to interact with browser tool                                  |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK, LlamaIndex                                  |

### Tutorial Architecture

In this tutorial we will describe how to use LlamaIndex with browser tool for automated web content analysis.

In our example we will send natural language instructions to the LlamaIndex agent to perform tasks on the Bedrock AgentCore browser.

### Tutorial Key Features

- **Browser Automation**: Automated web navigation and content extraction
- **LlamaIndex Analysis**: AI-powered content analysis with Bedrock Claude
- **Screenshot Capture**: Full page and viewport screenshots
- **Simple Interface**: Single-command execution
- **Universal Compatibility**: Works with any website and custom prompts

## Prerequisites

To execute this tutorial you will need:
* Python 3.12+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* LlamaIndex SDK
* Playwright for browser automation

## 1. Environment Setup

In [ ]:
# Set up Python 3.12 virtual environment
!python3.12 --version
!python3.12 -m venv venv
!source venv/bin/activate && python --version

In [ ]:
# Install dependencies
!pip install --force-reinstall -U -r requirements.txt --quiet

print("✅ All dependencies installed successfully!")

## 2. Setup and Imports

In [ ]:
# Import required libraries
import asyncio
import json
from datetime import datetime
from pathlib import Path

# LlamaIndex imports
from llama_index.llms.bedrock_converse import BedrockConverse

# Browser and Playwright imports
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright

# Utilities
from rich.console import Console
from rich.panel import Panel
import boto3

console = Console()

print("✅ All libraries imported successfully!")

## 3. Browser Automation Class Implementation

In [ ]:
class BrowserAutomationWithLlamaIndex:
    """
    Browser automation with LlamaIndex for web content analysis
    """
    
    def __init__(self, region="us-east-1"):
        self.region = region
        self.browser_client = None
        self.results_dir = Path("analysis_results")
        self.results_dir.mkdir(exist_ok=True)
        
    async def run_analysis(self, prompt, starting_page):
        """
        Main function that runs the complete browser analysis workflow
        """
        console.print(
            Panel(
                f"[bold cyan]LlamaIndex Browser Analysis[/bold cyan]\n\n"
                f"🎯 Task: {prompt}\n"
                f"🌐 Starting Page: {starting_page}\n"
                f"📁 Results: {self.results_dir}",
                title="Browser Analysis Session",
                border_style="blue",
            )
        )
        
        try:
            # Step 1: Initialize browser session
            console.print("\n[cyan]🚀 Initializing browser session...[/cyan]")
            
            # Create browser session
            self.browser_client = browser_session(self.region).__enter__()
            ws_url, headers = self.browser_client.generate_ws_headers()
            console.print(f"[green]✅ Browser session: {self.browser_client.session_id}[/green]")
            
            # Step 2: Set up Playwright automation
            console.print("\n[cyan]🤖 Setting up Playwright automation...[/cyan]")
            
            # Create LlamaIndex LLM for analysis
            llm = BedrockConverse(
                model="anthropic.claude-3-haiku-20240307-v1:0",
                region_name=self.region,
                temperature=0.1,
                max_tokens=4000
            )
            
            console.print("[green]✅ LlamaIndex LLM ready[/green]")
            
            # Step 3: Execute browser automation
            console.print(f"\n[cyan]🎬 Starting browser automation...[/cyan]")
            
            # Generate timestamp for this analysis
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            # Use Playwright to automate the browser session
            async with async_playwright() as p:
                browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
                context = browser.contexts[0] if browser.contexts else await browser.new_context()
                page = context.pages[0] if context.pages else await context.new_page()
                
                console.print(f"[green]✅ Connected to browser session[/green]")
                
                # Navigate to starting page
                console.print(f"[yellow]🌐 Navigating to: {starting_page}[/yellow]")
                await page.goto(starting_page, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)  # Brief pause for page load
                
                # Extract page content
                console.print(f"[yellow]📄 Extracting page content...[/yellow]")
                page_title = await page.title()
                page_content = await page.inner_text('body')
                
                console.print(f"[green]✅ Page loaded: {page_title}[/green]")
                console.print(f"[green]📄 Content extracted ({len(page_content)} characters)[/green]")
                
                # Take screenshots
                console.print(f"[yellow]📸 Taking screenshots...[/yellow]")
                
                # Full page screenshot
                full_screenshot = self.results_dir / f"analysis_{timestamp}_full.png"
                await page.screenshot(path=str(full_screenshot), full_page=True)
                
                # Viewport screenshot  
                viewport_screenshot = self.results_dir / f"analysis_{timestamp}_viewport.png"
                await page.screenshot(path=str(viewport_screenshot), full_page=False)
                
                console.print(f"[green]📸 Screenshots saved[/green]")
                
                # Use LlamaIndex to analyze the content
                console.print(f"[cyan]🤖 Analyzing content with LlamaIndex...[/cyan]")
                
                analysis_prompt = f"""
                Analyze this web page content and complete the requested task:
                
                Page Title: {page_title}
                URL: {starting_page}
                Task: {prompt}
                
                Page Content:
                {page_content[:3000]}
                
                Please provide a detailed analysis focusing on the specific task requested.
                Extract the exact information requested and present it clearly.
                """
                
                response = await llm.acomplete(analysis_prompt)
                result = response.text
            
            # Step 4: Save results and provide summary
            # Save the result
            result_data = {
                "timestamp": timestamp,
                "prompt": prompt,
                "starting_page": starting_page,
                "page_title": page_title,
                "content_length": len(page_content),
                "agent_response": str(result),
                "session_id": self.browser_client.session_id,
                "screenshots": {
                    "full_page": str(full_screenshot),
                    "viewport": str(viewport_screenshot)
                },
                "success": True
            }
            
            result_file = self.results_dir / f"analysis_{timestamp}.json"
            with open(result_file, 'w', encoding='utf-8') as f:
                json.dump(result_data, f, indent=2, ensure_ascii=False)
            
            # Also save as text for easy reading
            text_file = self.results_dir / f"analysis_{timestamp}.txt"
            with open(text_file, 'w', encoding='utf-8') as f:
                f.write(f"Browser Analysis Results\n")
                f.write(f"{'='*50}\n\n")
                f.write(f"Timestamp: {timestamp}\n")
                f.write(f"Task: {prompt}\n")
                f.write(f"Starting Page: {starting_page}\n")
                f.write(f"Session ID: {self.browser_client.session_id}\n\n")
                f.write(f"Agent Response:\n")
                f.write(f"{'-'*30}\n")
                f.write(f"{result}\n")
            
            # Display results
            console.print(f"\n[bold green]✅ Analysis Complete![/bold green]")
            console.print(f"📋 Results saved: {result_file}")
            console.print(f"📄 Text summary: {text_file}")
            console.print(f"📸 Full page screenshot: {full_screenshot}")
            console.print(f"📸 Viewport screenshot: {viewport_screenshot}")
            
            console.print(f"\n[bold cyan]🎯 Agent Response:[/bold cyan]")
            console.print(Panel(str(result), title="Analysis Results", border_style="green"))
            
            return result_data
            
        except Exception as e:
            console.print(f"[red]❌ Error during analysis: {e}[/red]")
            import traceback
            console.print(f"[dim]{traceback.format_exc()}[/dim]")
            return {
                "error": str(e),
                "success": False
            }
        
        finally:
            # Cleanup
            try:
                if self.browser_client:
                    self.browser_client.stop()
                    console.print("[green]✅ Browser session cleaned up[/green]")
            except Exception as e:
                console.print(f"[yellow]⚠️ Cleanup warning: {e}[/yellow]")

print("✅ BrowserAutomationWithLlamaIndex class defined!")

## 4. Initialize the System

In [ ]:
# Get AWS region
boto_session = boto3.Session()
region = boto_session.region_name or "us-east-1"

# Initialize the browser analysis system
analyzer = BrowserAutomationWithLlamaIndex(region=region)

console.print("🚀 Browser Analysis System with LlamaIndex ready!")
console.print("\n📋 System capabilities:")
console.print("   ✅ Automated browser navigation")
console.print("   ✅ Playwright automation")
console.print("   ✅ Screenshot capture (full page + viewport)")
console.print("   ✅ LlamaIndex + Bedrock Claude 3 Haiku")
console.print("   ✅ Content extraction and analysis")
console.print("   ✅ Universal website compatibility")
console.print(f"\n🎯 Using AWS region: {region}")
console.print("\n🎬 Ready for browser automation!")

## 5. Usage Examples

Run these examples to see browser automation in action!

### Example 1: Stock Analysis (Tested ✅)

In [ ]:
# Analyze Apple stock - automated browser navigation and content extraction
# This example has been tested and works perfectly
result = await analyzer.run_analysis(
    prompt="Find and extract the current stock price, market cap, and P/E ratio",
    starting_page="https://stockanalysis.com/stocks/aapl/"
)

### Example 2: News Headlines Extraction

In [ ]:
# Extract news headlines with automated browser
result = await analyzer.run_analysis(
    prompt="Extract the top 3 news headlines and provide a brief summary of each",
    starting_page="https://news.ycombinator.com"
)

### Example 3: Financial Data Extraction

In [ ]:
# Extract financial data with automated browser
result = await analyzer.run_analysis(
    prompt="Extract Tesla's current stock price, market cap, and recent performance metrics",
    starting_page="https://finance.yahoo.com/quote/TSLA"
)

### Example 4: GitHub Trending Analysis

In [ ]:
# Analyze GitHub trending repositories with automated browser
result = await analyzer.run_analysis(
    prompt="What are the top 5 trending repositories and what technologies are they using?",
    starting_page="https://github.com/trending"
)

## 6. Command Line Usage

You can also use the standalone script for single-command execution:

```bash
# Stock analysis (tested and working ✅)
python browser_automation_with_llamaindex.py --prompt "Find and extract the current stock price, market cap, and P/E ratio" --starting-page "https://stockanalysis.com/stocks/aapl/"

# News extraction  
python browser_automation_with_llamaindex.py --prompt "Extract the top 3 news headlines" --starting-page "https://news.ycombinator.com"

```

## 7. Results

Each analysis creates files in the `analysis_results` directory:
- `analysis_TIMESTAMP.json` - Complete structured results with metadata
- `analysis_TIMESTAMP.txt` - Human-readable text summary
- `analysis_TIMESTAMP_full.png` - Full page screenshot
- `analysis_TIMESTAMP_viewport.png` - Viewport screenshot

## What happened behind the scenes?

* The BrowserAutomationWithLlamaIndex class automatically manages browser session creation and automation
* You configured the system to use LlamaIndex with Bedrock Claude 3 Haiku for intelligent content analysis
* The system created a browser session and used Playwright for automated web navigation
* LlamaIndex took your natural language instructions and performed web automation tasks
* The agent navigated to the target page, extracted content, captured screenshots, and analyzed the information using AI
* All results are saved locally for review and analysis

## What You'll Get

- **Automated browser navigation** with intelligent content extraction
- **AI-powered analysis** of any website content
- **Screenshots** (full page + viewport)
- **Structured results** saved as JSON and text files

The system provides headless browser automation with LlamaIndex and Claude 3 Haiku for intelligent content analysis!

## Test Results

After running the examples above, you'll see results like this:

**Example: Apple Stock Analysis (AAPL)**
- Current Stock Price: $255.46
- Market Cap: $3.79 Trillion  
- P/E Ratio: 38.86
- Screenshots: ✅ Captured and saved
- Analysis: ✅ Accurate extraction and analysis